In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from rtnls_inference.ensembles import ClassificationEnsemble

# Fundus quality classification

This example uses the local quality release, whose embedded configuration names the classes **good**, **usable**, and **bad**. No inference overrides are needed.

Run the notebook from this directory. Adjust `QUALITY_MODEL` to your local release file. The examples below show a single loaded image and then a collection of image files.

In [ ]:
# Sample raw fundus photographs, relative to this notebook's directory.
INPUT_DIR = Path("../samples/fundus/original")

# Use "preprocessed" for images that already have the canonical fundus crop.
INPUT_SPACE = "raw"

# Prefer GPU 1 when available, then GPU 0, otherwise use CPU.
DEVICE = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0" if torch.cuda.is_available() else "cpu")

# Number of images processed together. Reduce this if GPU memory is limited.
BATCH_SIZE = 2

# Local model file containing the class mapping in its embedded configuration.
QUALITY_MODEL = Path("../../rtnls_vascx_models/quality/quality.pt")

In [ ]:
image_paths = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
)
print(f"Found {len(image_paths)} images; using {DEVICE}.")

ensemble = ClassificationEnsemble.from_torchscript(QUALITY_MODEL).to(DEVICE).eval()

## 1. Single-input inference

Preprocess → make a batch → predict → postprocess. The example starts with a loaded RGB uint8 NumPy image. You can also pass a file path, or a uint8 tensor shaped `(3, height, width)`, in the `image` field.

`result["label"]` is the predicted name. `result["class_id"]` is its index, and `result["probabilities"]` contains softmax probabilities in class-index order: good, usable, bad. The original averaged logits remain available in `result["output"]`. Postprocessing does not run the model again.

In [ ]:
image_path = image_paths[0]
with Image.open(image_path) as image:
    raw_image = np.array(image.convert("RGB"))

# 1. Preprocess the loaded RGB image. No ID is needed for an in-memory result.
item = ensemble.preprocess_item({"image": raw_image}, input_space=INPUT_SPACE)

# 2. Make a batch of size 1.
batch = ensemble.make_batch([item])

# 3. Run the ensemble.
predictions = ensemble.predict_batch(batch)

# 4. Postprocess using the same batch's input context.
result = ensemble.postprocess_batch(predictions, batch)[0]

print("Predicted quality:", result["label"])
print("Probability:", float(result["probabilities"][result["class_id"]]))

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(result["preprocessed_image"])
plt.title(f"{image_path.name}: {result['label']}")
plt.axis("off")
plt.show()

## 2. Batch inference

Create the input records and a dataloader, then call `predict_dataloader` to return a compact table. It uses the same preprocessing and postprocessing as the single-image example. No destination is supplied, so no prediction files are written.

The grid shows each input image with its predicted quality label and the probability assigned to that label.

Path inputs get IDs from their filename stems, which must be unique. IDs are optional for already-loaded images when returning results in memory.

Input records are ordinary dictionaries. Creating the dataloader validates them once; image files are loaded when the loader is iterated. Predictions and geometry are dictionaries too.


In [ ]:
# 1. Create records. Each filename stem becomes the image's ID.
data = [{"image": path} for path in image_paths]

# 2. Create the inference dataloader.
loader = ensemble.make_inference_dataloader(
    data,
    input_space=INPUT_SPACE,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

In [ ]:
# 3. Return a compact prediction table in memory. No destination means no files.
results = ensemble.predict_dataloader(loader)
results


In [ ]:
# Arrange all sample images in a grid, with three images per row.
ncols = 3
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, path, result in zip(axes.flat, image_paths, results.to_dict("records")):
    with Image.open(path) as image:
        ax.imshow(image.convert("RGB"))
    probability = result[f"probability_{result['label']}"]
    ax.set_title(f"{result['id']}\n{result['label']} ({probability:.1%})")
fig.tight_layout()
plt.show()
